# Corrosion Data Analysis — NIST CORR-DATA

**Portfolio analysis:** classify corrosion-resistance ratings (A/B/C/D) using material, environment, concentration, and temperature information from the NIST CORR-DATA database.

This notebook presents the final analysis in a clean, reproducible sequence. Ambiguous source values are not silently converted into invented measurements, and validation is designed to show both random-split performance and generalization to an unseen source reference.

## 1. Setup and load data

Download the official NIST CSV and place it at `data/CORR-DATA_Database.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('data/CORR-DATA_Database.csv')
print('Raw dataset shape:', df.shape)

## 2. Clean text fields and extract the target

Only clear A/B/C/D ratings are used. Ambiguous entries are excluded from the classification target rather than forcing them into a class.

In [ ]:
for column in ['Environment', 'Material Group', 'Material Family', 'Material']:
    df[column + '_clean'] = df[column].astype('string').str.strip()

df['material_group_clean'] = df['Material Group_clean'].replace({'MIscellaneous': 'Miscellaneous'})

rating_text = df['Rate (mm/yr) or Rating'].astype('string').str.strip()
clear_rating = (
    rating_text.str.fullmatch(r'[ABCD]', na=False)
    | rating_text.str.match(r'^[ABCD]\s*\([^/)]*\)\s*$', na=False)
)
df['corrosion_rating'] = rating_text.where(clear_rating).str.extract(r'^([ABCD])', expand=False)
rated_df = df[df['corrosion_rating'].notna()].copy()

classification_df = rated_df[[
    'material_group_clean', 'Material Family_clean', 'Material_clean',
    'Environment_clean', 'corrosion_rating'
]].dropna().copy()

classification_df.to_csv('data/processed/corrosion_classification_baseline.csv', index=False)
print('Rated observations:', len(rated_df))
print('Classification dataset:', classification_df.shape)
print('\nTarget distribution:')
print(classification_df['corrosion_rating'].value_counts())

## 3. Create safe concentration and temperature features

Single numeric values are retained. Clear numeric ranges are represented by their midpoint. Qualitative values are left unavailable. Concentration values above 100 are not reinterpreted because the source column is labelled Vol %.

In [ ]:
# Temperature
temperature_text = rated_df['Temperature (deg C)'].astype('string').str.strip()
rated_df['temperature_numeric'] = pd.to_numeric(temperature_text, errors='coerce')
temp_range = temperature_text.str.fullmatch(r'-?\d+(?:\.\d+)?\s*-\s*-?\d+(?:\.\d+)?', na=False)
rated_df['temperature_min_c'] = rated_df['temperature_numeric']
rated_df['temperature_max_c'] = rated_df['temperature_numeric']
temp_parts = temperature_text[temp_range].str.split('-', expand=True)
rated_df.loc[temp_range, 'temperature_min_c'] = pd.to_numeric(temp_parts[0], errors='coerce')
rated_df.loc[temp_range, 'temperature_max_c'] = pd.to_numeric(temp_parts[1], errors='coerce')
rated_df['temperature_mid_c'] = (rated_df['temperature_min_c'] + rated_df['temperature_max_c']) / 2

# Concentration
concentration_text = rated_df['Concentration (Vol %)'].astype('string').str.strip()
rated_df['concentration_min'] = pd.to_numeric(concentration_text, errors='coerce')
rated_df['concentration_max'] = rated_df['concentration_min']
invalid_single = (rated_df['concentration_min'] < 0) | (rated_df['concentration_min'] > 100)
rated_df.loc[invalid_single, ['concentration_min', 'concentration_max']] = np.nan
conc_range = concentration_text.str.fullmatch(r'\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?', na=False)
conc_parts = concentration_text[conc_range].str.split('-', expand=True)
cmin = pd.to_numeric(conc_parts[0], errors='coerce')
cmax = pd.to_numeric(conc_parts[1], errors='coerce')
valid = (cmin >= 0) & (cmin <= 100) & (cmax >= 0) & (cmax <= 100)
valid_idx = cmin[valid].index
rated_df.loc[valid_idx, 'concentration_min'] = cmin.loc[valid_idx]
rated_df.loc[valid_idx, 'concentration_max'] = cmax.loc[valid_idx]
rated_df['concentration_mid'] = (rated_df['concentration_min'] + rated_df['concentration_max']) / 2

print('Usable temperature midpoints:', rated_df['temperature_mid_c'].notna().sum())
print('Usable concentration midpoints:', rated_df['concentration_mid'].notna().sum())
print('Concentration values above 100 excluded from numeric feature:', ((pd.to_numeric(concentration_text, errors='coerce') > 100)).sum())

## 4. Train three comparable Random Forest models

All three models use the same 80/20 stratified split. Missing numerical values are filled using the training-set median only, and missingness indicators are retained.

In [ ]:
base_features = ['material_group_clean', 'Material Family_clean', 'Material_clean', 'Environment_clean']
numeric_features = ['concentration_mid', 'temperature_mid_c']

model_df = rated_df[base_features + numeric_features + ['corrosion_rating']].dropna(subset=base_features).copy()
X = model_df.drop(columns='corrosion_rating')
y = model_df['corrosion_rating']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

def prepare_features(X_train, X_test, numeric_cols):
    X_train = X_train.copy()
    X_test = X_test.copy()
    categorical = base_features
    encoder = OneHotEncoder(handle_unknown='ignore')
    encoded_train = encoder.fit_transform(X_train[categorical])
    encoded_test = encoder.transform(X_test[categorical])
    numeric_train = X_train[numeric_cols].copy()
    numeric_test = X_test[numeric_cols].copy()
    for col in numeric_cols:
        numeric_train[col + '_missing'] = numeric_train[col].isna().astype(int)
        numeric_test[col + '_missing'] = numeric_test[col].isna().astype(int)
        median = numeric_train[col].median()
        numeric_train[col] = numeric_train[col].fillna(median)
        numeric_test[col] = numeric_test[col].fillna(median)
    numeric_names = []
    for col in numeric_cols:
        numeric_names.extend([col, col + '_missing'])
    final_train = hstack([encoded_train, numeric_train[numeric_names].to_numpy(dtype=np.float64)])
    final_test = hstack([encoded_test, numeric_test[numeric_names].to_numpy(dtype=np.float64)])
    return final_train, final_test

def train_and_score(train_matrix, test_matrix, y_train, y_test):
    model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    model.fit(train_matrix, y_train)
    predictions = model.predict(test_matrix)
    return model, predictions, accuracy_score(y_test, predictions), f1_score(y_test, predictions, average='macro'), f1_score(y_test, predictions, average='weighted')

# Baseline
base_train, base_test = prepare_features(X_train, X_test, [])
model1, pred1, acc1, macro1, weighted1 = train_and_score(base_train, base_test, y_train, y_test)

# Concentration
conc_train, conc_test = prepare_features(X_train, X_test, ['concentration_mid'])
model2, pred2, acc2, macro2, weighted2 = train_and_score(conc_train, conc_test, y_train, y_test)

# Concentration + temperature
full_train, full_test = prepare_features(X_train, X_test, ['concentration_mid', 'temperature_mid_c'])
model3, pred3, acc3, macro3, weighted3 = train_and_score(full_train, full_test, y_train, y_test)

comparison = pd.DataFrame({
    'Model': ['Baseline', 'Baseline + Concentration', 'Baseline + Concentration + Temperature'],
    'Accuracy': [acc1, acc2, acc3],
    'Macro F1': [macro1, macro2, macro3],
    'Weighted F1': [weighted1, weighted2, weighted3]
})
print(comparison.to_string(index=False))
comparison.to_csv('results/model_comparison.csv', index=False)

## 5. Random-split performance

The final random-split model is Model 3, which combines categorical material/environment features with concentration and temperature.

In [ ]:
print(f'Final model accuracy: {acc3:.2%}')
print('\nClassification report:')
print(classification_report(y_test, pred3))

cm3 = confusion_matrix(y_test, pred3, labels=['A', 'B', 'C', 'D'])
print('Confusion matrix:')
print(cm3)

ConfusionMatrixDisplay(cm3, display_labels=['A', 'B', 'C', 'D']).plot()
plt.title('Model 3 Corrosion Rating Confusion Matrix')
plt.tight_layout()
plt.savefig('results/model3_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Feature importance

Feature importance helps identify which inputs the Random Forest relied on most. It is a model diagnostic and does not prove physical causation.

In [ ]:
feature_names = list(model3.feature_importances_.shape and [])
feature_names = list(model3.feature_importances_)
print('Feature-importance vector length:', len(feature_names))

# Recover names from the fitted encoder and numeric columns used above.
encoded_names = list(OneHotEncoder(handle_unknown='ignore').fit(X_train[base_features]).get_feature_names_out(base_features))
all_names = encoded_names + ['concentration_mid', 'concentration_mid_missing', 'temperature_mid_c', 'temperature_mid_c_missing']
importance_df = pd.DataFrame({'Feature': all_names, 'Importance': model3.feature_importances_}).sort_values('Importance', ascending=False).head(20)
importance_df.to_csv('results/model3_feature_importance_top20.csv', index=False)
print(importance_df.to_string(index=False))

importance_df.sort_values('Importance').plot.barh(x='Feature', y='Importance', legend=False, figsize=(9, 7))
plt.title('Top 20 Model 3 Feature Importances')
plt.tight_layout()
plt.savefig('results/model3_feature_importance_top20.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Strict unseen-reference validation

Reference #253 contains 79.35% of all rated observations. It is therefore held out completely to test whether a model trained on other source references generalizes to this dominant but unseen reference.

This is intentionally a much harder test than the random split.

In [ ]:
reference_df = rated_df[base_features + numeric_features + ['corrosion_rating', 'Reference #']].dropna(subset=base_features).copy()
reference_df['Reference #'] = pd.to_numeric(reference_df['Reference #'], errors='coerce')
reference_df = reference_df[reference_df['Reference #'].notna()].copy()

train_ref = reference_df[reference_df['Reference #'] != 253].copy()
test_ref = reference_df[reference_df['Reference #'] == 253].copy()

X_ref_train = train_ref.drop(columns=['corrosion_rating', 'Reference #'])
y_ref_train = train_ref['corrosion_rating']
X_ref_test = test_ref.drop(columns=['corrosion_rating', 'Reference #'])
y_ref_test = test_ref['corrosion_rating']

ref_train_matrix, ref_test_matrix = prepare_features(
    X_ref_train, X_ref_test, ['concentration_mid', 'temperature_mid_c']
)
ref_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
ref_model.fit(ref_train_matrix, y_ref_train)
ref_pred = ref_model.predict(ref_test_matrix)
ref_accuracy = accuracy_score(y_ref_test, ref_pred)
ref_macro = f1_score(y_ref_test, ref_pred, average='macro')

print('Reference-based training observations:', len(train_ref))
print('Reference #253 test observations:', len(test_ref))
print(f'Reference #253 accuracy: {ref_accuracy:.2%}')
print(f'Reference #253 Macro F1: {ref_macro:.3f}')
print('\nClassification report:')
print(classification_report(y_ref_test, ref_pred))

cm_ref = confusion_matrix(y_ref_test, ref_pred, labels=['A', 'B', 'C', 'D'])
ConfusionMatrixDisplay(cm_ref, display_labels=['A', 'B', 'C', 'D']).plot()
plt.title('Unseen Reference #253 Confusion Matrix')
plt.tight_layout()
plt.savefig('results/reference_253_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. What explains the validation gap?

Reference #253 has many material and environment categories that are not present in the training references. This diagnostic measures that category shift without changing the test data.

In [ ]:
training_materials = set(train_ref['Material_clean'])
training_environments = set(train_ref['Environment_clean'])

test_ref['material_unseen'] = ~test_ref['Material_clean'].isin(training_materials)
test_ref['environment_unseen'] = ~test_ref['Environment_clean'].isin(training_environments)

print(f'Unseen-material observations: {test_ref["material_unseen"].sum()} ({test_ref["material_unseen"].mean():.2%})')
print(f'Unseen-environment observations: {test_ref["environment_unseen"].sum()} ({test_ref["environment_unseen"].mean():.2%})')
print(f'Both unseen: {(test_ref["material_unseen"] & test_ref["environment_unseen"]).sum()} ({(test_ref["material_unseen"] & test_ref["environment_unseen"]).mean():.2%})')

known_material = ~test_ref['material_unseen']
known_environment = ~test_ref['environment_unseen']
print(f'Accuracy when material was seen in training: {accuracy_score(y_ref_test[known_material], ref_pred[known_material]):.2%}')
print(f'Accuracy when material was unseen in training: {accuracy_score(y_ref_test[~known_material], ref_pred[~known_material]):.2%}')
print(f'Accuracy when environment was seen in training: {accuracy_score(y_ref_test[known_environment], ref_pred[known_environment]):.2%}')
print(f'Accuracy when environment was unseen in training: {accuracy_score(y_ref_test[~known_environment], ref_pred[~known_environment]):.2%}')

## 9. Final interpretation

The random stratified split reaches **78.35% accuracy and 0.701 Macro F1**, while the strict unseen Reference #253 holdout reaches **39.54% accuracy and 0.26 Macro F1**.

The difference is an important finding rather than a result to hide. It shows that random-split performance reflects the observed dataset distribution and may not represent performance on a substantially different source reference.

The analysis therefore demonstrates both model building and validation awareness: preprocessing is leakage-conscious, source values are handled conservatively, and generalization is tested beyond a simple random split.